# <font color='blue'> Extração, Transformação e Carregamento dos microdados dos Dados dos Exames Laboratoriais da Pesquisa Nacional de Saúde - PNS 2013</font>

### Engenharia de Dados para Análise de Indicadores de Risco em Diabetes (Modelagem para BI)

**Breve explicação dessa limpeza inicial e seleção de variáveis**
* Este **notebook** documenta a primeira etapa do fluxo de dados para a construção de um painel analítico voltado ao estudo do `Diabetes Mellitus` utilizando os **Dados dos Exames Laboratoriais da Pesquisa Nacional de Saúde 2013**, realizada no ano de 2013 e revisada em 2020. Como a base de microdados da PNS 2013 é extensa, importá-la em seu estado bruto gera lentidão e ruído analítico. Esta limpeza atua como um filtro estrutural focado na doença: o código reduz a dimensionalidade do dataset selecionando estritamente os fatores relevantes e indicadores de risco associados ao diabetes. O escopo foi delimitado para capturar métricas laboratoriais (Hemoglobina Glicada, Colesterol), parâmetros vitais (Pressão Arterial, Circunferência da Cintura), histórico de tratamento/internação e as dimensões sociodemográficas dos registros.

**Objetivos do Projeto (Nesta etapa)**
* O objetivo central é preparar os dados epidemiológicos para a construção de um dashboard interativo no **Power BI**, utilizando as melhores práticas de modelagem relacional. Ao traduzir a complexidade dos códigos do IBGE e tratar os valores inválidos na raiz via Python, garante-se a integridade matemática da base. O foco é arquitetar os dados para permitir, na etapa visual, uma análise profunda dos casos de diabetes, mapeando a prevalência da doença, cruzando perfis de risco e avaliando o impacto do estilo de vida nos indicadores clínicos populacionais.

**Solução Proposta**
* Executar a leitura dos microdados brutos aplicando os delimitadores textuais corretos.

* Filtrar o dataframe isolando exclusivamente as variáveis de interesse clínico, metabólico e sociodemográfico relacionadas à saúde do diabético.

* Padronizar a nomenclatura das colunas, substituindo os códigos alfanuméricos originais por descrições clínicas claras em formato Snake Case (ex: de `Z034` para `Hemoglobina_Glicada`). Utilizando o dicionário disponibilizado pelo IBGE.

* Tratar inconsistências de preenchimento e criar um identificador único (`ID_Entrevistado`) para garantir a granularidade exata de um registro por indivíduo da subamostra.

* Modelar os dados em arquitetura Star Schema, fatiando a base plana em uma Tabela Fato (`métricas clínicas`) e Tabelas Dimensão (`filtros descritivos`).

**Resultados esperados**
* A geração de um pacote de arquivos estruturados e independentes (`.csv`) contemplando as tabelas Fato e Dimensões. Este modelo relacional higienizado será a fonte de dados consumida pelo Power BI, performance no processamento das medidas DAX e permitindo a livre exploração visual dos cruzamentos entre os fatores de risco, ocorrência e consequências do diabetes.

## EXTRAÇÃO DA BASE DE DADOS

In [1]:
# Importação das bibliotecas que serão utilizadas
import pandas as pd
import numpy as np
from rich import print, inspect
from pathlib import Path 

In [2]:
# Especificando o caminho do arquivo EXAMES_PNS_2013_BRUTA.csv
caminho_arquivo_pns = Path(r"data\raw\EXAMES_PNS_2013_BRUTA.csv") #insira o caminho conforme o seu diretório

# Confirmação de que o arquivo foi encontrado dentro do caminho informado
if caminho_arquivo_pns.exists():
    print("Arquivo encontrado")
else:
    print("Não encontrado")

Arquivo encontrado

In [3]:
# Criando um data frame a partir do arquivo bruto
# Encoding='utf-8' --> garantir a compatibilidade de caracteres em diferentes
# Decimal=',' --> evitar erros relacionados a números decimais
# Low_memory=False --> evita avisos de tipagem mista ao carregar bases grandes.
df_pns_base = pd.read_csv(caminho_arquivo_pns, encoding='utf-8', decimal=',',low_memory=False)

# Monstrando o cabeçalho do data frame bruto
print(df_pns_base.head(1))

Z001  Z002  C008  Z003  z004   z005  Z006  Z007  Z008  Z009  ...  X02507  \
0   1.0  18.0  18.0   4.0  49.3  166.8  4.62  13.5  42.5  91.9  ...     2.0   

   X02508  X02509  X02510  F012  W00303  W00407  W00408  regiao  peso_lab  
0     2.0     2.0     2.0     2    67.7    95.0    52.0     1.0      0.46  

[1 rows x 509 columns]

### MAPEAMENTO DE VARIÁVEIS

In [4]:
# Criando um dicionário de Mapeamento para os nomes das colunas que serão utilizadas na análise
mapeamento_colunas = {
    'Z001': 'Sexo', 
    'C008': 'Idade', 
    'Z003': 'Cor_Raca', 
    'z004': 'Peso_kg', 
    'z005': 'Altura_cm',
    'Z031': 'Colesterol_Total', 
    'Z034': 'Hemoglobina_Glicada', 
    'I001': 'Possui_Plano_Saude',
    'J004': 'Motivo_Impedimento_Atividades', 
    'P034': 'Pratica_Exercicio', 
    'P035': 'Dias_Exercicio_Semana',
    'Q002': 'Avaliacao_Saude', 
    'Q030': 'Diagnostico_Diabetes', 
    'Q031': 'Idade_Diagnostico_Diabetes',
    'Q03401': 'Uso_Medicamento_Diabetes',
    'Q056': 'Internacao_por_Diabetes', 
    'Q060': 'Diagnostico_Hipertensao',
    'VDD004': 'Nivel_Instrucao', 
    'W00303': 'Circunferencia_Cintura_cm', 
    'W00407': 'Pressao_Sistolica_mmHg',
    'W00408': 'Pressao_Diastolica_mmHg', 
    'regiao': 'Regiao', 
    'peso_lab': 'Peso_Lab',
    'Q035': 'Uso_Insulina', 
    'Q05501': 'Problema_Vista',
    'Q05502': 'Infarto',
    'Q05503': 'AVC',
    'Q05504': 'Problema_Circulatorio',
    'Q05505': 'Problema_Rins',
    'Q05506': 'Ulcera_Pes',
    'Q05507': 'Amputacao',
    'Q05508': 'Coma_Diabetico'
}

inspect (mapeamento_colunas)

╭──────────────────────────── <class 'dict'> ────────────────────────────╮
│ dict() -> new empty dictionary                                         │
│ dict(mapping) -> new dictionary initialized from a mapping object's    │
│     (key, value) pairs                                                 │
│ dict(iterable) -> new dictionary initialized as if via:                │
│     d = {}                                                             │
│     for k, v in iterable:                                              │
│         d[k] = v                                                       │
│ dict(**kwargs) -> new dictionary initialized with the name=value pairs │
│     in the keyword argument list.  For example:  dict(one=1, two=2)    │
│                                                                        │
│ ╭────────────────────────────────────────────────────────────────────╮ │
│ │ {                                                                  │ │
│ │ │   'Z001': 'Sexo',                                                │ │
│ │ │   'C008': 'Idade',                                               │ │
│ │ │   'Z003': 'Cor_Raca',                                            │ │
│ │ │   'z004': 'Peso_kg',                                             │ │
│ │ │   'z005': 'Altura_cm',                                           │ │
│ │ │   'Z031': 'Colesterol_Total',                                    │ │
│ │ │   'Z034': 'Hemoglobina_Glicada',                                 │ │
│ │ │   'I001': 'Possui_Plano_Saude',                                  │ │
│ │ │   'J004': 'Motivo_Impedimento_Atividades',                       │ │
│ │ │   'P034': 'Pratica_Exercicio',                                   │ │
│ │ │   ... +22                                                        │ │
│ │ }                                                                  │ │
│ ╰────────────────────────────────────────────────────────────────────╯ │
│                                                                        │
│ 35 attribute(s) not shown. Run inspect(inspect) for options.           │
╰────────────────────────────────────────────────────────────────────────╯

### STAGING AREA

In [5]:
# Definindo o caminho de saída para a Staging Area (dentro da pasta raw)
# .parent para pegar a pasta onde está o arquivo original ou definimos o caminho relativo
caminho_staging = caminho_arquivo_pns.parent / "PNS_2013_STAGING.csv"

# Renomeando as colunas baseado no mapeamento realizado
df_pns_base = df_pns_base.rename(columns=mapeamento_colunas)

# Criando uma variável que recebe uma lista com os valores das colunas mapeadas
colunas_projeto = list(mapeamento_colunas.values())

# Criando o data frame com as colunas selecionadas
df_variaveis = df_pns_base[colunas_projeto].copy()

# Salvando a cópia física na pasta RAW
# index=False evita criar uma coluna extra de números
# encoding='utf-8-sig' garante que o Excel abra os acentos corretamente
df_variaveis.to_csv(caminho_staging, index=False, sep=';', encoding='utf-8-sig')

print(f"Staging Area salva com sucesso em: {caminho_staging}")
print(df_variaveis.head(1))

Staging Area salva com sucesso em: C:\Users\avila\Desktop\Analise_PNS_2013\data\raw\PNS_2013_STAGING.csv

Sexo  Idade  Cor_Raca  Peso_kg  Altura_cm  Colesterol_Total  \
0   1.0   18.0       4.0     49.3      166.8             132.0   

   Hemoglobina_Glicada  Possui_Plano_Saude  Motivo_Impedimento_Atividades  \
0                 5.35                   2                            NaN   

   Pratica_Exercicio  ...  Peso_Lab  Uso_Insulina  Problema_Vista  Infarto  \
0                2.0  ...      0.46           NaN             NaN      NaN   

   AVC  Problema_Circulatorio  Problema_Rins  Ulcera_Pes  Amputacao  \
0  NaN                    NaN            NaN         NaN        NaN   

   Coma_Diabetico  
0             NaN  

[1 rows x 32 columns]

## TRANSFORMAÇÃO DOS DADOS

### CRIAÇÃO DE PRIMARY KEY

In [6]:
# Cria uma coluna com com um ID único para cada entrevistado 
df_variaveis.insert(0, 'ID_Entrevistado', range(1, len(df_variaveis) + 1))

print(f"Chave primária {df_variaveis['ID_Entrevistado']} criada")

Chave primária 0          1
1          2
2          3
3          4
4          5
        ... 
8947    8948
8948    8949
8949    8950
8950    8951
8951    8952
Name: ID_Entrevistado, Length: 8952, dtype: int64 criada

### TRADUÇÃO DOS DADOS

In [7]:
# Exibindo toas as colunas do data frame afim de verificar se alguma coluna foi excluída
pd.set_option('display.max_columns', None)
print(df_variaveis.head())

ID_Entrevistado  Sexo  Idade  Cor_Raca  Peso_kg  Altura_cm  \
0                1   1.0   18.0       4.0     49.3      166.8   
1                2   1.0   18.0       4.0     66.1      170.0   
2                3   1.0   18.0       1.0     71.7      176.7   
3                4   1.0   18.0       4.0     67.4      160.5   
4                5   1.0   19.0       4.0     84.2      169.2   

   Colesterol_Total  Hemoglobina_Glicada  Possui_Plano_Saude  \
0             132.0                 5.35                   2   
1             136.0                 4.58                   2   
2             222.0                 5.44                   2   
3             117.0                 5.44                   2   
4             149.0                 5.20                   2   

   Motivo_Impedimento_Atividades  Pratica_Exercicio  Dias_Exercicio_Semana  \
0                            NaN                2.0                    NaN   
1                            NaN                1.0                    7.0   
2                            NaN                1.0                    0.0   
3                            NaN                1.0                    5.0   
4                            NaN                2.0                    NaN   

   Avaliacao_Saude  Diagnostico_Diabetes  Idade_Diagnostico_Diabetes  \
0              NaN                   NaN                         NaN   
1              3.0                   3.0                         NaN   
2              3.0                   3.0                         NaN   
3              NaN                   NaN                         NaN   
4              3.0                   3.0                         NaN   

   Uso_Medicamento_Diabetes  Internacao_por_Diabetes  Diagnostico_Hipertensao  \
0                       NaN                      NaN                      NaN   
1                       NaN                      NaN                      2.0   
2                       NaN                      NaN                      2.0   
3                       NaN                      NaN                      NaN   
4                       NaN                      NaN                      2.0   

   Nivel_Instrucao  Circunferencia_Cintura_cm  Pressao_Sistolica_mmHg  \
0                2                       67.7                    95.0   
1                4                       79.5                   121.5   
2                4                       84.5                   126.0   
3                1                       82.1                   122.5   
4                4                      100.2                   107.4   

   Pressao_Diastolica_mmHg  Regiao  Peso_Lab  Uso_Insulina  Problema_Vista  \
0                     52.0     1.0      0.46           NaN             NaN   
1                     68.5     2.0      1.04           NaN             NaN   
2                     76.0     5.0      1.18           NaN             NaN   
3                     72.0     2.0      1.36           NaN             NaN   
4                     64.4     1.0      0.51           NaN             NaN   

   Infarto  AVC  Problema_Circulatorio  Problema_Rins  Ulcera_Pes  Amputacao  \
0      NaN  NaN                    NaN            NaN         NaN        NaN   
1      NaN  NaN                    NaN            NaN         NaN        NaN   
2      NaN  NaN                    NaN            NaN         NaN        NaN   
3      NaN  NaN                    NaN            NaN         NaN        NaN   
4      NaN  NaN                    NaN            NaN         NaN        NaN   

   Coma_Diabetico  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN

In [8]:
# Traduzindo os dados seguindo o dicionário disponibilizado pelo IBGE
# Criando dicionários para tradução
dict_sim_nao = {'1': 'Sim', '2': 'Não'}
dict_sexo = {'1': 'Masculino', '2': 'Feminino'}
dict_raca = {'1': 'Branca', '2': 'Preta', '3': 'Amarela', '4': 'Parda', '5': 'Indígena', '9': 'Ignorado'}
dict_regiao = {'1': 'Norte', '2': 'Nordeste', '3': 'Sudeste', '4': 'Sul', '5': 'Centro-Oeste'}
dict_saude = {'1': 'Muito boa', '2': 'Boa', '3': 'Regular', '4': 'Ruim', '5': 'Muito ruim'}
dict_instrucao = {'1': 'Sem instrução', '2': 'Fundamental Incompleto', '3': 'Fundamental Completo', 
                  '4': 'Médio Incompleto', '5': 'Médio Completo', '6': 'Superior Incompleto', '7': 'Superior Completo'}

# Criando uma lista de todas as colunas de Sim/Não
cols_sim_nao = [
    'Diagnostico_Diabetes', 'Diagnostico_Hipertensao', 'Uso_Medicamento_Diabetes', 
    'Internacao_por_Diabetes', 'Possui_Plano_Saude', 'Pratica_Exercicio', 'Uso_Insulina',
    'Problema_Vista', 'Infarto', 'AVC', 'Problema_Circulatorio', 'Problema_Rins', 
    'Ulcera_Pes', 'Amputacao', 'Coma_Diabetico'
]

print("Traduções criadas")

Traduções criadas

In [9]:
# Criando uma variável que armazena todas as colunas que precisam de tradução
todas_para_traduzir = ['Sexo', 'Cor_Raca', 'Regiao', 'Nivel_Instrucao', 'Avaliacao_Saude'] + cols_sim_nao

# Usando a variável que armazenou as colunas que precisam de tradução 
# transforma tudo em texto, tira espaços e remove '.0' escondidos
for col in todas_para_traduzir:
    df_variaveis[col] = df_variaveis[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)


#Aplicando as traduções textuais
df_variaveis['Sexo'] = df_variaveis['Sexo'].map(dict_sexo)
df_variaveis['Cor_Raca'] = df_variaveis['Cor_Raca'].map(dict_raca)
df_variaveis['Regiao'] = df_variaveis['Regiao'].map(dict_regiao)
df_variaveis['Nivel_Instrucao'] = df_variaveis['Nivel_Instrucao'].map(dict_instrucao)
df_variaveis['Avaliacao_Saude'] = df_variaveis['Avaliacao_Saude'].map(dict_saude)

#Aplicando a tradução nas colunas de Sim/Não
for col in cols_sim_nao:
    df_variaveis[col] = df_variaveis[col].map(dict_sim_nao)

print("Traduções aplicadas")

Traduções aplicadas

### TRATAMENTO DOS NULOS

In [10]:
# Definindo as colunas de complicações realacionadas ao DM
# que podem possuir NULOS
cols_complica = [
    'Uso_Medicamento_Diabetes', 'Internacao_por_Diabetes', 'Uso_Insulina', 'Problema_Vista', 
    'Infarto', 'AVC', 'Problema_Circulatorio', 'Problema_Rins', 'Ulcera_Pes', 'Amputacao', 'Coma_Diabetico'
]

# Substitui os valores NULOS nas colunas por um valor textual "Não se aplica"
df_variaveis[cols_complica] = df_variaveis[cols_complica].fillna('Não se aplica')

# Definindo as colunas de categorias gerais que podem possuir NULOS
cols_categoricas_gerais = ['Motivo_Impedimento_Atividades', 'Possui_Plano_Saude', 'Avaliacao_Saude', 'Pratica_Exercicio']

# Substitui os valores NULOS nas colunas por um valor textual "Não se aplica"
df_variaveis[cols_categoricas_gerais] = df_variaveis[cols_categoricas_gerais].fillna('Não informado')

df_variaveis.to_csv('df_variaveis.csv', index=False, sep=';', decimal=',')

print("Tratamento de Nulos Realizado")

Tratamento de Nulos Realizado

# <font color='blue'>Arquitetura Star Schema (Modelagem Dimensional)</font>

## Implementação do modelo relacional otimizado para Business Intelligence

* Nesta etapa, o notebook realiza a transição da base de dados plana para uma estrutura de `Star Schema`. O objetivo é garantir a máxima performance analítica no `Power BI`, organizando as variáveis em uma estrutura lógica que separa métricas quantitativas de atributos descritivos.

**Estruturação da Tabela Fato e Dimensões**

* A Tabela Fato (`Fato_Individual_Lab`) foi consolidada para armazenar o grão da análise: o registro clínico de cada indivíduo. Nela, concentram-se as métricas de exames (HbA1c, Colesterol), medidas antropométricas (IMC, Pressão Arterial) e, fundamentalmente, o Peso Laboratorial, garantindo que qualquer cálculo posterior respeite a representatividade estatística da subamostra.

* As variáveis qualitativas foram fatiadas em Tabelas Dimensão independentes, permitindo filtragens multidimensionais eficientes. A `Dim_Geografia` organiza o território; a `Dim_Sociodemografia` agrupa perfis de cor, sexo e instrução; a `Dim_Estilo_Vida_Acesso` isola comportamentos e posse de plano de saúde; enquanto as dimensões de `Perfil_Clinico` e `Dim_Complicacoes` detalham a jornada e a gravidade da doença.

**Integridade Relacional e Persistência**

* A integridade do modelo é assegurada pela criação de surrogate keys sintéticas, geradas a partir das combinações únicas de atributos de cada dimensão via `drop_duplicates()` e `reset_index()`, permitindo que o motor de BI execute cruzamentos sem redundância de dados. O processo finaliza com a exportação de cada entidade em arquivos `.csv` independentes, utilizando codificação `utf-8-sig` e delimitadores compatíveis com sistemas de gestão de bancos de dados.

**Resultados do Modelo**

* O resultado é um banco de dados relacional pronto para consumo, onde a lógica de negócio já está pré-processada na fonte. Isso reduz a complexidade de modelagem dentro da ferramenta de visualização e garante que as correlações entre fatores de risco e subnotificação sejam calculadas de forma ágil e precisa.

## CÓPIA DA STAGING AREA

In [11]:
# Criação de uma cópia de segurança da Staging Area para não alterar a original
df_modelagem = df_variaveis.copy()

print("Cópia de segurança da Staging Area criada com sucesso")

Cópia de segurança da Staging Area criada com sucesso

## COLUNAS DERIVADAS

### LÓGICA DE STATUS GLICÊMICO REAL

In [12]:
# Criando a lógica de Status Glicêmico Real no Python
condicoes_glicemia = [
    (df_modelagem['Hemoglobina_Glicada'] >= 6.5) | (df_modelagem['Uso_Medicamento_Diabetes'] == 'Sim') | (df_modelagem['Diagnostico_Diabetes'] == 'Sim'),
    (df_modelagem['Hemoglobina_Glicada'] >= 5.7) & (df_modelagem['Hemoglobina_Glicada'] < 6.5),
    (df_modelagem['Hemoglobina_Glicada'] > 0) & (df_modelagem['Hemoglobina_Glicada'] < 5.7)
]

rotulos_glicemia = ['Diabetes', 'Pré-diabetes', 'Normal']

df_modelagem['Status_Glicemico_Real'] = np.select(condicoes_glicemia, rotulos_glicemia, default='Não Analisado')

### CÁLCULO IMC

In [13]:
# Convertendo para número de forma segura (trocando vírgula por ponto)
peso_num = pd.to_numeric(df_modelagem['Peso_kg'].astype(str).str.replace(',', '.'), errors='coerce')
altura_m = pd.to_numeric(df_modelagem['Altura_cm'].astype(str).str.replace(',', '.'), errors='coerce') / 100

# Criando a coluna utilizando o cálculo de IMC: peso (kg) / (altura²)
df_modelagem['IMC'] = (peso_num / (altura_m ** 2)).round(2)

print("Cálculo de IMC criado")

Cálculo de IMC criado

### DISCRETIZAÇÃO - FAIXA ETÁRIA

In [14]:
# Convertendo para número de forma segura (trocando vírgula por ponto)
idade_num = pd.to_numeric(df_modelagem['Idade'].astype(str).str.replace(',', '.'), errors='coerce')

# Definindo discretização de faixa-etária
bins_idade = [0, 29, 45, 59, 150]
labels_idade = ['18 a 29 anos', '30 a 45 anos', '46 a 59 anos', '60 anos ou mais']

# Utilizando pd.cut para binning baseado discretização realizada acima
df_modelagem['Faixa_Etaria'] = pd.cut(idade_num, bins=bins_idade, labels=labels_idade)

# Tratamento de resultados nulos 
df_modelagem['Faixa_Etaria'] = df_modelagem['Faixa_Etaria'].astype(str).replace('nan', 'Não informado')

print("Dicretização de faixa etária criada")

Dicretização de faixa etária criada

### DISCRETIZAÇÃO - DIAS DE EXERCÍCIO

In [15]:
# Convertendo para número de forma segura (trocando vírgula por ponto)
dias_num = pd.to_numeric(df_modelagem['Dias_Exercicio_Semana'].astype(str).str.replace(',', '.'), errors='coerce')

# Definindo discretização de dias de exercício
bins_dias = [-1, 0, 2, 4, 7]
labels_dias = ['0 dias (Sedentário)', '1 a 2 dias (Leve)', '3 a 4 dias (Moderado)', '5 a 7 dias (Intenso)']

# Utilizando pd.cut para binning baseado discretização realizada acima
df_modelagem['Faixa_Dias_Exercicio'] = pd.cut(dias_num, bins=bins_dias, labels=labels_dias)

# Tratamento de resultados nulos 
df_modelagem['Faixa_Dias_Exercicio'] = df_modelagem['Faixa_Dias_Exercicio'].astype(str).replace('nan', 'Não informado')
print("Discretização de dias de exercício criado")

Discretização de dias de exercício criado

### RÓTULO SEMÂNTICO - PERFIL DE MORBIDADE

In [16]:
# Utilizando np.select() para aplicar regras de negócio em cascata
# Condições graves
cond_grave = (df_modelagem['Amputacao'] == 'Sim') | (df_modelagem['AVC'] == 'Sim') | (df_modelagem['Infarto'] == 'Sim')

# Condições moderadas
cond_moderado = (df_modelagem['Problema_Vista'] == 'Sim') | (df_modelagem['Problema_Rins'] == 'Sim') | (df_modelagem['Ulcera_Pes'] == 'Sim') | (df_modelagem['Problema_Circulatorio'] == 'Sim') | (df_modelagem['Coma_Diabetico'] == 'Sim') | (df_modelagem['Internacao_por_Diabetes'] == 'Sim')

# Condições de diabético em tratamento
cond_tratamento = (df_modelagem['Uso_Medicamento_Diabetes'] == 'Sim') | (df_modelagem['Uso_Insulina'] == 'Sim')

# Condições de diabético base
cond_diabetico_base = (df_modelagem['Diagnostico_Diabetes'] == 'Sim')

# Agrupa todas as condições em uma única lista
condicoes = [cond_grave, cond_moderado, cond_tratamento, cond_diabetico_base]

# Agrupa todos os rotulos em uma única lista
rotulos = ['3. Grave (Eventos Agudos/Amputação)', '2. Moderado (Complicações/Internação)', '1. Tratamento Base (Meds/Insulina)', '0. Diabético (Sem tratamento reportado)']

# Default para entrevistados sem diagnóstico de diabetes
df_modelagem['Perfil_Morbidade'] = np.select(condicoes, rotulos, default = 'Não diabéticos')

print("Rótulo semântico de perfil de morbidade criado com sucesso")

Rótulo semântico de perfil de morbidade criado com sucesso

## CRIAÇÃO DAS DIMENSÕES

### SOCIODEMOGRAFIA

In [17]:
# Definindo as colunas que serão utilizadas para criar a dimensão
cols_Sociodemografia = ['Sexo', 'Cor_Raca', 'Nivel_Instrucao', 'Faixa_Etaria']

# Definindo a dimensão a partir da cópia da staging e colunas que serão utilizadas 
dim_Sociodemografia = df_modelagem[cols_Sociodemografia].drop_duplicates().reset_index(drop=True)

# Criando o ID_Sociodemografia 
dim_Sociodemografia.insert(0, 'ID_Sociodemografia', dim_Sociodemografia.index + 1)


df_modelagem = df_modelagem.merge(dim_Sociodemografia, on=cols_Sociodemografia, how='left')

print("Dim_Sociodemografia criada")

Dim_Sociodemografia criada

### PERFIL CLÍNICO

In [18]:
# Definindo as colunas que serão utilizadas para criar a dimensão
cols_perfil_clinico = [
    'Diagnostico_Diabetes', 'Diagnostico_Hipertensao', 'Avaliacao_Saude', 
    'Uso_Medicamento_Diabetes', 'Internacao_por_Diabetes', 'Motivo_Impedimento_Atividades'
]

# Definindo a dimensão a partir da cópia da staging e colunas que serão utilizadas 
dim_Perfil_Clinico = df_modelagem[cols_perfil_clinico].drop_duplicates().reset_index(drop=True)

# Cirando o ID_Perfil_clinico
dim_Perfil_Clinico.insert(0, 'ID_Perfil_clinico', dim_Perfil_Clinico.index + 1)

df_modelagem = df_modelagem.merge(dim_Perfil_Clinico, on=cols_perfil_clinico, how='left')

print("dim_Perfil_Clinico criada")

dim_Perfil_Clinico criada

### ESTILO DE VIDA

In [19]:
# Definindo as colunas que serão utilizadas para criar a dimensão
cols_estilo_vida = ['Pratica_Exercicio', 'Faixa_Dias_Exercicio', 'Possui_Plano_Saude']

# Definindo a dimensão a partir da cópia da staging e colunas que serão utilizadas 
dim_Estilo_Vida_Acesso = df_modelagem[cols_estilo_vida].drop_duplicates().reset_index(drop=True)

# Criando o ID_Estilo_Vida
dim_Estilo_Vida_Acesso.insert(0, 'ID_Estilo_Vida', dim_Estilo_Vida_Acesso.index + 1)

df_modelagem = df_modelagem.merge(dim_Estilo_Vida_Acesso, on=cols_estilo_vida, how='left')

print("dim_Estilo_Vida_Acesso criada")

dim_Estilo_Vida_Acesso criada

### COMPLICAÇÕES DA DIABETES

In [20]:
# Definindo as colunas que serão utilizadas para criar a dimensão
cols_Complicacoes_Diabetes = [
    'Perfil_Morbidade', 'Uso_Insulina', 'Problema_Vista', 'Infarto', 'AVC', 
    'Problema_Circulatorio', 'Problema_Rins', 'Ulcera_Pes', 'Amputacao', 'Coma_Diabetico'
]

# Definindo a dimensão a partir da cópia da staging e colunas que serão utilizadas 
dim_Complicacoes_Diabetes = df_modelagem[cols_Complicacoes_Diabetes].drop_duplicates().reset_index(drop=True)

# Criando o ID_Complicacoes
dim_Complicacoes_Diabetes.insert(0, 'ID_Complicacoes', dim_Complicacoes_Diabetes.index + 1)

df_modelagem = df_modelagem.merge(dim_Complicacoes_Diabetes, on=cols_Complicacoes_Diabetes, how='left')

print("Dim_Complicacoes_Diabetes criada")

Dim_Complicacoes_Diabetes criada

### GEOGRAFIA

In [21]:
# Definindo as colunas que serão utilizadas para criar a dimensão
cols_geografia = ['Regiao']

# Definindo a dimensão a partir da cópia da staging e colunas que serão utilizadas 
dim_Geografia = df_modelagem[cols_geografia].drop_duplicates().sort_values('Regiao').reset_index(drop=True)

# Criando o ID_Geografia
dim_Geografia.insert(0, 'ID_Geografia', dim_Geografia.index + 1)

df_modelagem = df_modelagem.merge(dim_Geografia, on=cols_geografia, how='left')

print("dim_Geografia criada")

dim_Geografia criada

## CRIAÇÃO DA FATO

### FATO INDIVIDUAL 

In [22]:
colunas_fato_final = [
    'ID_Entrevistado', 
    'ID_Sociodemografia', 
    'ID_Perfil_clinico', 
    'ID_Complicacoes', 
    'ID_Estilo_Vida', 
    'ID_Geografia',
    'Idade', 
    'Idade_Diagnostico_Diabetes', 
    'Peso_kg', 
    'Altura_cm', 
    'IMC', 
    'Circunferencia_Cintura_cm', 
    'Pressao_Sistolica_mmHg', 
    'Pressao_Diastolica_mmHg', 
    'Colesterol_Total', 
    'Hemoglobina_Glicada',
    'Status_Glicemico_Real',
    'Peso_Lab'
]
fato_Individual_Lab = df_modelagem[colunas_fato_final]

print("fato_Individual_Lab criada")

#Verifica se existe algum nulo nas FKs
fks = ['ID_Sociodemografia','ID_Perfil_clinico',
       'ID_Complicacoes','ID_Estilo_Vida','ID_Geografia']
nulos = {fk: fato_Individual_Lab[fk].isna().sum() for fk in fks}
print("Nulos por FK:", nulos)

fato_Individual_Lab criada

Nulos por FK:
{
    'ID_Sociodemografia': np.int64(0),
    'ID_Perfil_clinico': np.int64(0),
    'ID_Complicacoes': np.int64(0),
    'ID_Estilo_Vida': np.int64(0),
    'ID_Geografia': np.int64(0)
}

## CARREGAMENTO

### EXPORTAÇÃO DAS TABELAS PARA SQLITE

In [23]:
# Define o caminho da pasta processed relativo ao notebook
output_dir = Path('\\Analise_PNS_2013\\data\\processed')

# Cria a pasta automaticamente se não existir
output_dir.mkdir(parents=True, exist_ok=True)

# Exportação
fato_Individual_Lab.to_csv(output_dir / 'Fato_Individual_Lab.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_Sociodemografia.to_csv(output_dir / 'Dim_Sociodemografia.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_Perfil_Clinico.to_csv(output_dir / 'Dim_Perfil_Clinico.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_Complicacoes_Diabetes.to_csv(output_dir / 'Dim_Complicacoes_Diabetes.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_Estilo_Vida_Acesso.to_csv(output_dir / 'Dim_Estilo_Vida_Acesso.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
dim_Geografia.to_csv(output_dir / 'Dim_Geografia.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

print(f"Arquivos exportados para: {output_dir.resolve()}")

Arquivos exportados para: C:\Users\avila\Desktop\Analise_PNS_2013\data\processed

**Resultados do Modelo**

O processo de modelagem gerou 6 artefatos estruturados prontos para consumo no Power BI:

| Artefato | Tipo | Linhas | Descrição |
|---|---|---|---|
| `Fato_Individual_Lab.csv` | Tabela Fato | 8.952 | Métricas clínicas + Peso_Lab + FKs |
| `Dim_Sociodemografia.csv` | Dimensão | ~271 | Sexo, Raça, Instrução, Faixa Etária |
| `Dim_Geografia.csv` | Dimensão | 5 | Região + coordenadas geográficas |
| `Dim_Perfil_Clinico.csv` | Dimensão | ~120 | Diagnósticos e percepção de saúde |
| `Dim_Complicacoes_Diabetes.csv` | Dimensão | ~50 | Perfil de morbidade e complicações |
| `Dim_Estilo_Vida_Acesso.csv` | Dimensão | ~40 | Exercício e acesso ao sistema de saúde |

Duas colunas derivadas de alto valor analítico foram criadas diretamente na camada de ETL,
pré-processando a lógica de negócio antes da ingestão no Power BI:

**`Status_Glicemico_Real`** — Classifica cada indivíduo com base em critério combinado
(ADA, 2013), confrontando o dado laboratorial com o autodeclarado:
```python
# Critério combinado: laboratorial + autodeclaração + uso de medicamento
SWITCH(TRUE(),
    HbA1c >= 6.5 || TomaRemedio = "Sim" || DeclarouDM = "Sim" → "Diabetes"
    HbA1c >= 5.7 && HbA1c < 6.5                               → "Pré-diabetes"
    HbA1c > 0 && HbA1c < 5.7                                  → "Normal"
                                                               → "Não Analisado"
)
```

**`Perfil_Morbidade`** — Estratifica os diabéticos em estágios clínicos via
`np.select()` em cascata, respeitando a hierarquia de severidade:
```python
condicoes = [cond_grave, cond_moderado, cond_tratamento, cond_diabetico_base]
rotulos   = ['3. Grave', '2. Moderado', '1. Tratamento Base', '0. Diabético Base']
# default → 'Não Diabético'
```

Essas duas variáveis são o núcleo analítico do dashboard e permitem quantificar
a **invisibilidade diagnóstica** (subnotificados identificados pelo confronto
HbA1c × autodeclaração) e estratificar a **progressão clínica** da doença
sem depender de cálculos complexos na camada DAX.